In [ ]:
import os
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

In [18]:
import pandas as pd

import os
os.chdir(r"C:\Users\USER\Downloads")

BR = pd.read_csv("BR.csv")  # BR.csv도 Downloads 폴더에 있는지 확인 필요

In [6]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

base = Path(r"C:\Users\USER\Desktop\따릉이")
use_dir = base / "use"

use_files = sorted(use_dir.glob("_Bike-Use*.csv"))

agg = defaultdict(lambda: {
    "총이용거리": 0,
    "총대여횟수": 0,
    "마지막대여일": pd.Timestamp("1900-01-01")
})

for i, f in enumerate(use_files):
    x = pd.read_csv(f, low_memory=False)

    x["대여일시_dt"] = pd.to_datetime(x["대여일시_dt"], errors="coerce")
    x["자전거번호"] = pd.to_numeric(x["자전거번호"], errors="coerce")
    x["이용거리(M)"] = pd.to_numeric(x["이용거리(M)"], errors="coerce").fillna(0)

    x = x.dropna(subset=["자전거번호", "대여일시_dt"])
    x["자전거번호"] = x["자전거번호"].astype(int)

    for bike, date, dist in zip(x["자전거번호"], x["대여일시_dt"], x["이용거리(M)"]):
        d = agg[bike]

        d["총이용거리"] += dist
        d["총대여횟수"] += 1

        if date > d["마지막대여일"]:
            d["마지막대여일"] = date

    print(f"[{i+1}/{len(use_files)}] 완료")
    rental_summary = pd.DataFrame([
    {
        "bike": bike,
        **vals
    }
    for bike, vals in agg.items()
])

[1/60] 완료
[2/60] 완료
[3/60] 완료
[4/60] 완료
[5/60] 완료
[6/60] 완료
[7/60] 완료
[8/60] 완료
[9/60] 완료
[10/60] 완료
[11/60] 완료
[12/60] 완료
[13/60] 완료
[14/60] 완료
[15/60] 완료
[16/60] 완료
[17/60] 완료
[18/60] 완료
[19/60] 완료
[20/60] 완료
[21/60] 완료
[22/60] 완료
[23/60] 완료
[24/60] 완료
[25/60] 완료
[26/60] 완료
[27/60] 완료
[28/60] 완료
[29/60] 완료
[30/60] 완료
[31/60] 완료
[32/60] 완료
[33/60] 완료
[34/60] 완료
[35/60] 완료
[36/60] 완료
[37/60] 완료
[38/60] 완료
[39/60] 완료
[40/60] 완료
[41/60] 완료
[42/60] 완료
[43/60] 완료
[44/60] 완료
[45/60] 완료
[46/60] 완료
[47/60] 완료
[48/60] 완료
[49/60] 완료
[50/60] 완료
[51/60] 완료
[52/60] 완료
[53/60] 완료
[54/60] 완료
[55/60] 완료
[56/60] 완료
[57/60] 완료
[58/60] 완료
[59/60] 완료
[60/60] 완료


In [21]:
rental_summary.to_csv(
    base / "rental_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

NameError: name 'rental_summary' is not defined

In [22]:
rental_summary = pd.read_csv(base / "rental_summary.csv")

NameError: name 'base' is not defined

rental_daily-> 전처리에서 저장했던 모든 Bike-Use 파일을 가져와서, 일별 집계

In [15]:
import pandas as pd
from pathlib import Path

base = Path(r"C:\Users\USER\Desktop\따릉이")
use_dir = base / "use"

use_files = sorted(use_dir.glob("*Bike-Use*.csv"))

daily_list = []

for i, f in enumerate(use_files):
    print(f"[{i+1}/{len(use_files)}] 처리중: {f.name}")

    x = pd.read_csv(f, low_memory=False)

    # 날짜 처리
    x["대여일시_dt"] = pd.to_datetime(x["대여일시_dt"], errors="coerce")
    x["date"] = x["대여일시_dt"].dt.floor("D")

    # 자전거번호 정리
    x["자전거번호"] = (
        x["자전거번호"]
        .astype(str)
        .str.replace("SPB-", "", regex=False)
        .str.extract(r"(\d+)")[0]
    )
    x["자전거번호"] = pd.to_numeric(x["자전거번호"], errors="coerce")

    # 거리
    x["이용거리(M)"] = pd.to_numeric(x["이용거리(M)"], errors="coerce").fillna(0)

    # 결측 제거
    x = x.dropna(subset=["자전거번호", "date"])
    x["자전거번호"] = x["자전거번호"].astype(int)

    # 🔥 핵심: 일별 집계
    daily = (
        x.groupby(["자전거번호", "date"])
        .agg(
            일일이용거리=("이용거리(M)", "sum"),
            일일대여횟수=("자전거번호", "count")
        )
        .reset_index()
    )

    daily_list.append(daily)

# 전체 합치기
rental_daily = pd.concat(daily_list, ignore_index=True)

print("완료:", rental_daily.shape)
rental_daily.head()

[1/60] 처리중: _Bike-Use0.csv
[2/60] 처리중: _Bike-Use1.csv


KeyboardInterrupt: 

In [ ]:
rental_daily.to_parquet(base / "rental_daily.parquet")
rental_daily = pd.read_parquet(base / "rental_daily.parquet")

NameError: name 'rental_daily' is not defined

In [16]:
import pandas as pd
from pathlib import Path

base = Path(r"C:\Users\USER\Desktop\따릉이")

rental_daily = pd.read_parquet(base / "rental_daily.parquet")

In [19]:
BR["date"] = pd.to_datetime(BR["date"], errors="coerce")
BR["bike"] = pd.to_numeric(BR["bike"], errors="coerce")

BR = BR.dropna(subset=["bike", "date"])
BR["bike"] = BR["bike"].astype(int)

fault_summary = (
    BR.groupby("bike")
    .agg(
        총고장횟수=("bike", "count"),
        마지막고장일=("date", "max"),
        주요고장유형=("type", lambda x: x.mode()[0] if len(x.mode()) > 0 else None)
    )
    .reset_index()
)

In [ ]:
rental_daily.to_pickle(base / "rental_daily.pkl")
# 여기를 fault_summary로 바꿔야되는디///
BR.to_pickle(base / "BR.pkl")

# 스냅샷

In [20]:
def build_snapshot_daily(rental_daily, BR, snapshot_date, window_days=30):

    snapshot_date = pd.Timestamp(snapshot_date)
    future_end = snapshot_date + pd.Timedelta(days=window_days)

    # 1. 과거 데이터
    rental_past = rental_daily[rental_daily["date"] <= snapshot_date]
    fault_past = BR[BR["date"] <= snapshot_date]

    if rental_past.empty:
        return pd.DataFrame()

    # 2. 대여 집계
    rental_agg = (
        rental_past
        .groupby("자전거번호")
        .agg(
            총이용거리=("일일이용거리", "sum"),
            총대여횟수=("일일대여횟수", "sum"),
            마지막대여일=("date", "max")
        )
        .reset_index()
    )

    # 3. 고장 집계
    if fault_past.empty:
        fault_agg = pd.DataFrame(columns=["자전거번호", "총고장횟수", "마지막고장일", "주요고장유형"])
    else:
        fault_type_mode = (
            fault_past
            .groupby("bike")["type"]
            .agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan)
            .rename("주요고장유형")
            .reset_index()
            .rename(columns={"bike": "자전거번호"})
        )

        fault_agg = (
            fault_past
            .groupby("bike")
            .agg(
                총고장횟수=("bike", "count"),
                마지막고장일=("date", "max")
            )
            .reset_index()
            .rename(columns={"bike": "자전거번호"})
            .merge(fault_type_mode, on="자전거번호", how="left")
        )

    # 4. merge
    snap = rental_agg.merge(fault_agg, on="자전거번호", how="left")

    # 5. 파생
    snap["총고장횟수"] = snap["총고장횟수"].fillna(0)

    snap["마지막고장후경과일"] = (
        snapshot_date - snap["마지막고장일"]
    ).dt.days.fillna(9999)

    snap["고장간격km"] = (
        (snap["총이용거리"] / 1000) /
        snap["총고장횟수"].replace(0, np.nan)
    ).fillna(99999)

    snap["주요고장유형"] = snap["주요고장유형"].fillna("고장없음")

    # 6. label
    future_fault_bikes = set(
        BR.loc[
            (BR["date"] > snapshot_date) &
            (BR["date"] <= future_end),
            "bike"
        ]
    )

    snap["label"] = snap["자전거번호"].isin(future_fault_bikes).astype(int)
    snap["snapshot_date"] = snapshot_date

    return snap

In [ ]:
train_dates = pd.date_range("2021-12-31", "2024-11-30", freq="M")
test_dates = pd.date_range("2025-01-31", "2025-11-30", freq="M")
# range에서 월말 데이터 불러오고반복된거 각 시점까지의 데이터만 사용해서 향후 30일 내 고장 여부(label)를 생성
train_data = pd.concat([
    build_snapshot_daily(rental_daily, BR, d, window_days=30)
    for d in train_dates
], ignore_index=True)

test_data = pd.concat([
    build_snapshot_daily(rental_daily, BR, d, window_days=30)
    for d in test_dates
    
], ignore_index=True)

print(train_data.shape)
print(test_data.shape)

print(train_data["snapshot_date"].min(), "~", train_data["snapshot_date"].max())
print(test_data["snapshot_date"].min(), "~", test_data["snapshot_date"].max())

print(train_data["label"].value_counts())
print(train_data["label"].value_counts(normalize=True))

print(test_data["label"].value_counts())
print(test_data["label"].value_counts(normalize=True))

train_data.to_pickle(base / "train_snapshot.pkl")
test_data.to_pickle(base / "test_snapshot.pkl")

C:\Users\USER\AppData\Local\Temp\ipykernel_6488\4288084223.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  train_dates = pd.date_range("2021-12-31", "2024-11-30", freq="M")
C:\Users\USER\AppData\Local\Temp\ipykernel_6488\4288084223.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  test_dates = pd.date_range("2025-01-31", "2025-11-30", freq="M")


NameError: name 'rental_daily' is not defined

In [34]:
train_data = pd.read_pickle(base / "train_snapshot.pkl")
test_data = pd.read_pickle(base / "test_snapshot.pkl")

print(train_data.columns)
print(train_data.head())

Index(['자전거번호', '총이용거리', '총대여횟수', '마지막대여일', '총고장횟수', '마지막고장일', '주요고장유형',
       '마지막고장후경과일', '고장간격km', 'label', 'snapshot_date'],
      dtype='object')
   자전거번호     총이용거리  총대여횟수     마지막대여일  총고장횟수     마지막고장일 주요고장유형  마지막고장후경과일  \
0     25  118680.0     31 2021-04-16    1.0 2021-07-13     체인      171.0   
1    291    4590.0      2 2021-01-03    0.0        NaT   고장없음     9999.0   
2    385  218000.0     59 2021-03-31    0.0        NaT   고장없음     9999.0   
3    811   29220.0     13 2021-02-16    0.0        NaT   고장없음     9999.0   
4    830   11010.0      5 2021-01-25    0.0        NaT   고장없음     9999.0   

     고장간격km  label snapshot_date  
0    118.68      0    2021-12-31  
1  99999.00      0    2021-12-31  
2  99999.00      0    2021-12-31  
3  99999.00      0    2021-12-31  
4  99999.00      0    2021-12-31  


# 파생변수 생성

In [4]:
# train 처리
train_data["마지막고장후경과일"] = train_data["마지막고장후경과일"].clip(upper=365)
train_data["고장간격km"] = train_data["고장간격km"].clip(upper=5000)
train_data["고장경험있음"] = (train_data["총고장횟수"] > 0).astype(int)

# test도 똑같이 처리
test_data["마지막고장후경과일"] = test_data["마지막고장후경과일"].clip(upper=365)
test_data["고장간격km"] = test_data["고장간격km"].clip(upper=5000)
test_data["고장경험있음"] = (test_data["총고장횟수"] > 0).astype(int)

# log feature
import numpy as np

train_data["총이용거리_log"] = np.log1p(train_data["총이용거리"])
train_data["총대여횟수_log"] = np.log1p(train_data["총대여횟수"])

test_data["총이용거리_log"] = np.log1p(test_data["총이용거리"])
test_data["총대여횟수_log"] = np.log1p(test_data["총대여횟수"])

print(train_data.columns)
print(train_data.head())

NameError: name 'train_data' is not defined

In [10]:
train_data["label"].value_counts(normalize=True)

label
0    0.789553
1    0.210447
Name: proportion, dtype: float64

In [5]:
def add_recent_features_fast(snap, rental_daily):
    snap = snap.copy()
    rental_daily = rental_daily.copy()

    snap["snapshot_date"] = pd.to_datetime(snap["snapshot_date"])
    rental_daily["date"] = pd.to_datetime(rental_daily["date"])

    result_list = []

    for d in snap["snapshot_date"].sort_values().unique():
        d = pd.Timestamp(d)

        one_snap = snap[snap["snapshot_date"] == d].copy()

        for days in [7, 30, 90]:
            tmp = rental_daily[
                (rental_daily["date"] <= d) &
                (rental_daily["date"] > d - pd.Timedelta(days=days))
            ]

            agg = (
                tmp.groupby("자전거번호")
                .agg(
                    **{
                        f"최근{days}일이용거리": ("일일이용거리", "sum"),
                        f"최근{days}일대여횟수": ("일일대여횟수", "sum"),
                        f"최근{days}일이용일수": ("date", "nunique")
                    }
                )
                .reset_index()
            )

            one_snap = one_snap.merge(agg, on="자전거번호", how="left")

        result_list.append(one_snap)

    out = pd.concat(result_list, ignore_index=True)

    recent_cols = [c for c in out.columns if c.startswith("최근")]
    out[recent_cols] = out[recent_cols].fillna(0)

    out["최근7일_30일대여비율"] = (
        out["최근7일대여횟수"] / out["최근30일대여횟수"].replace(0, np.nan)
    ).fillna(0)

    out["최근7일_30일거리비율"] = (
        out["최근7일이용거리"] / out["최근30일이용거리"].replace(0, np.nan)
    ).fillna(0)

    out["마지막대여후경과일"] = (
        out["snapshot_date"] - out["마지막대여일"]
    ).dt.days.fillna(365).clip(upper=365)

    return out

In [6]:
train_data = add_recent_features_fast(train_data, rental_daily)
test_data = add_recent_features_fast(test_data, rental_daily)

print(train_data.columns)
print(train_data.head())

NameError: name 'train_data' is not defined

In [7]:
train_data.to_pickle(base / "train_snapshot_v2.pkl")
test_data.to_pickle(base / "test_snapshot_v2.pkl")

NameError: name 'train_data' is not defined

# 이것만 실행

In [ ]:
from pathlib import Path

base = Path("C:/Users/USER/Desktop/따릉이")  # 너 저장했던 폴더 경로
train_data = pd.read_pickle(base / "train_snapshot_v2.pkl")
test_data = pd.read_pickle(base / "test_snapshot_v2.pkl")

In [14]:
rental_daily.to_pickle(base / "rental_daily.pkl")
BR.to_pickle(base / "BR.pkl")

NameError: name 'rental_daily' is not defined

# 모델링

AUC ≈ 0.63, precision 낮음, recall 높음->모델이 못 배운 게 아니라 “입력 정보가 부족한 상태”

In [9]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report

features = [
    "총이용거리",
    "총대여횟수",
    "총고장횟수",
    "마지막고장후경과일",
    "고장간격km",
    "고장경험있음",

    "최근7일이용거리",
    "최근7일대여횟수",
    "최근30일이용거리",
    "최근30일대여횟수",
    "최근90일이용거리",
    "최근90일대여횟수",

    "최근7일_30일대여비율",
    "최근7일_30일거리비율",
    "마지막대여후경과일"
]

X_train = train_data[features]
y_train = train_data["label"]

X_test = test_data[features]
y_test = test_data["label"]

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss"
)
model.fit(X_train, y_train)

# 예측
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

AUC: 0.7025564392013391
              precision    recall  f1-score   support

           0       0.93      0.56      0.70    465762
           1       0.21      0.73      0.32     72250

    accuracy                           0.58    538012
   macro avg       0.57      0.65      0.51    538012
weighted avg       0.83      0.58      0.65    538012



In [28]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

for t in np.arange(0.1, 0.9, 0.1):
    y_pred_t = (y_prob >= t).astype(int)
    print(
        f"threshold={t:.1f}",
        "precision=", round(precision_score(y_test, y_pred_t), 3),
        "recall=", round(recall_score(y_test, y_pred_t), 3),
        "f1=", round(f1_score(y_test, y_pred_t), 3)
    )

threshold=0.1 precision= 0.155 recall= 0.996 f1= 0.268
threshold=0.2 precision= 0.173 recall= 0.972 f1= 0.293
threshold=0.3 precision= 0.184 recall= 0.95 f1= 0.308
threshold=0.4 precision= 0.191 recall= 0.914 f1= 0.315
threshold=0.5 precision= 0.205 recall= 0.73 f1= 0.321
threshold=0.6 precision= 0.251 recall= 0.223 f1= 0.236
threshold=0.7 precision= 0.318 recall= 0.04 f1= 0.071
threshold=0.8 precision= 0.373 recall= 0.001 f1= 0.002


In [29]:
test_result = test_data.copy()
test_result["고장확률"] = y_prob

risk_rank = test_result.sort_values("고장확률", ascending=False)

top_100 = risk_rank.head(100)
top_500 = risk_rank.head(500)

top_100[["자전거번호", "snapshot_date", "고장확률", "label"]].head()

,자전거번호,snapshot_date,고장확률,label
59030,39426,2025-02-28,0.867242,1
532870,74822,2025-11-30,0.846526,1
374987,69812,2025-08-31,0.846046,1
326088,70534,2025-07-31,0.845613,1
405621,50747,2025-09-30,0.843870,1


In [30]:
for n in [100, 500, 1000, 5000]:
    top_n = risk_rank.head(n)
    hit_rate = top_n["label"].mean()
    print(f"Top {n} 실제 고장 비율:", round(hit_rate, 3))

Top 100 실제 고장 비율: 0.37
Top 500 실제 고장 비율: 0.392
Top 1000 실제 고장 비율: 0.397
Top 5000 실제 고장 비율: 0.334


In [31]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

         feature  importance
9      최근30일대여횟수    0.677726
14     마지막대여후경과일    0.095272
7       최근7일대여횟수    0.078711
12  최근7일_30일대여비율    0.029698
3      마지막고장후경과일    0.020552
8      최근30일이용거리    0.018880
0          총이용거리    0.017204
2          총고장횟수    0.015635
11     최근90일대여횟수    0.011232
1          총대여횟수    0.010361
10     최근90일이용거리    0.007808
4         고장간격km    0.007676
6       최근7일이용거리    0.005367
13  최근7일_30일거리비율    0.003877
5         고장경험있음    0.000000


# 스트림릿

In [32]:
import joblib
from pathlib import Path

base = Path(r"C:\Users\USER\Desktop\따릉이")

# 모델 저장
joblib.dump(model, base / "xgb_model.pkl")

# 피처 목록도 저장 (Streamlit에서 동일하게 써야 함)
import json
with open(base / "features.json", "w") as f:
    json.dump(features, f, ensure_ascii=False)

print("저장 완료!")
print(base / "xgb_model.pkl")

저장 완료!
C:\Users\USER\Desktop\따릉이\xgb_model.pkl


In [10]:
import pandas as pd
df = pd.read_pickle(r"C:\Users\USER\Desktop\따릉이\test_snapshot_v2.pkl")
print(df.columns.tolist())
print(df.shape)

['자전거번호', '총이용거리', '총대여횟수', '마지막대여일', '총고장횟수', '마지막고장일', '주요고장유형', '마지막고장후경과일', '고장간격km', 'label', 'snapshot_date', '고장경험있음', '총이용거리_log', '총대여횟수_log', '최근7일이용거리', '최근7일대여횟수', '최근7일이용일수', '최근30일이용거리', '최근30일대여횟수', '최근30일이용일수', '최근90일이용거리', '최근90일대여횟수', '최근90일이용일수', '최근7일_30일대여비율', '최근7일_30일거리비율', '마지막대여후경과일']
(538012, 26)


In [11]:
print(df["snapshot_date"].unique())

<DatetimeArray>
['2025-01-31 00:00:00', '2025-02-28 00:00:00', '2025-03-31 00:00:00',
 '2025-04-30 00:00:00', '2025-05-31 00:00:00', '2025-06-30 00:00:00',
 '2025-07-31 00:00:00', '2025-08-31 00:00:00', '2025-09-30 00:00:00',
 '2025-10-31 00:00:00', '2025-11-30 00:00:00']
Length: 11, dtype: datetime64[ns]
